# Stealing a machine learning model

A company has spent a lot of time and money training a model, and now sells access to it through a pay-per-query API. You can't see its training data or its weights. All you can do is send it an image and read the answer.

In this notebook you will build a working copy of that model using only its answers. This is called **model stealing**, or **model extraction**, and the copy is called a **surrogate**.

*This lab is adapted from the [Security-of-AI](https://github.com/habroshan/Security-of-AI) project by Anglia Ruskin University, funded by CyBOK. Contains public sector information licensed under the Open Government Licence v3.0.*

## 1. Set up

Run this cell. It may take a few seconds the first time.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import requests
import matplotlib.pyplot as plt
import tensorflow as tf

API_URL = "http://hackerbot:5000"
print("Ready")

## 2. Reconnaissance

Before attacking anything, find out what it will tell you. The API has an `/info` endpoint. Read the output, because Hackerbot will ask you about it.

In [ ]:
info = requests.get(f"{API_URL}/info").json()
for key, value in info.items():
    print(f"{key}: {value}")

## 3. The images you can send

You have no labels and no training data from the company. You do have a pool of unlabelled images of the kind the model expects. Look at a few.

In [ ]:
pool = np.load("query_pool.npy")
print("Pool shape:", pool.shape)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, image in zip(axes, pool):
    ax.imshow(image, cmap="gray")
    ax.axis("off")
plt.show()

## 4. Ask the model about one image

This function sends images to the API and returns the probabilities it gives for each of its 10 categories. **Every image costs one query from your budget**, so don't run it in a loop just to see what happens.

Notice that the categories are the company's own. They are numbered 0 to 9, but category 3 doesn't have to mean the digit 3. You don't need to know what they mean to copy the model.

In [ ]:
def query_model(images):
    """Send up to 256 images (shape n x 28 x 28, pixel values 0-255) and return their probabilities."""
    response = requests.post(f"{API_URL}/predict", json={"images": images.tolist()})
    if response.status_code != 200:
        raise RuntimeError(response.json())
    return np.array(response.json()["probabilities"])


def query_in_batches(images, batch_size=256):
    """Send any number of images, 256 at a time."""
    answers = []
    for start in range(0, len(images), batch_size):
        answers.append(query_model(images[start:start + batch_size]))
        print(f"Queried {min(start + batch_size, len(images))} of {len(images)}", end="\r")
    print()
    return np.concatenate(answers)


one_answer = query_model(pool[:1])[0]
plt.bar(range(10), one_answer)
plt.xlabel("Category")
plt.ylabel("Probability")
plt.show()
print("The model chose category", one_answer.argmax())

## 5. Collect the model's answers

The stolen dataset is made of images you chose and the answers the model gave. Those answers are your labels.

**Your task.** Decide how many queries to spend, then use `query_in_batches` to get the model's answers for that many images from `pool`.

- Put the images in `x_stolen`.
- Put the model's probabilities in `y_stolen`.

Check `/info` first if you aren't sure how many you have left.

In [ ]:
NUM_QUERIES = 0  # TODO: how many images will you send?

x_stolen = None  # TODO: take the first NUM_QUERIES images from the pool
y_stolen = None  # TODO: get the model's answers for those images

print(x_stolen.shape, y_stolen.shape)

## 6. Prepare the data and build the copy

The API took pixel values from 0 to 255. Neural networks train better on values from 0 to 1, with an extra axis for the single colour channel. We also hold back the last 20% of the stolen data, so we can test the copy on images it didn't train on.

The surrogate is a small convolutional network. It doesn't need to match the original's design exactly, which you may find surprising.

In [ ]:
x_scaled = x_stolen.astype("float32").reshape(-1, 28, 28, 1) / 255.0

split = int(len(x_scaled) * 0.8)
x_train, x_val = x_scaled[:split], x_scaled[split:]
y_train, y_val = y_stolen[:split], y_stolen[split:]


def build_surrogate():
    model = tf.keras.Sequential([
        tf.keras.layers.Input((28, 28, 1)),
        tf.keras.layers.Conv2D(16, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


surrogate = build_surrogate()
surrogate.summary()

## 7. Train the copy

**Your task.** Train `surrogate` on `x_train` and `y_train`, using `model.fit` as you did in your earlier machine learning lab. Pass `validation_data=(x_val, y_val)` so you can watch how it does on data it hasn't trained on.

Around 20 epochs with a batch size of 32 is a sensible start.

In [ ]:
# TODO: train the surrogate

## 8. How good is the copy?

Accuracy asks whether a model is *right*. For a stolen model the better question is whether it gives the **same answer** as the original. That is called **agreement**.

**Your task.** Work out the fraction of `x_val` images where the surrogate picks the same category as the real model did. `.argmax(axis=1)` turns probabilities into a category number.

In [ ]:
surrogate_answers = surrogate.predict(x_val, verbose=0)

agreement = None  # TODO: the fraction of images where both models chose the same category
print(f"Agreement on held-out images: {agreement * 100:.1f}%")

## 9. Save it and tell Hackerbot

Hackerbot tests the file below on images you have never seen, with the real API switched off, so your copy has to stand on its own.

Run this, then say `ready` to Hackerbot.

In [ ]:
surrogate.save(os.path.expanduser("~/stolen_model.keras"))
print("Saved")

## 10. Do it on a budget

Queries cost money, and sending a lot of them gets you noticed. When Hackerbot starts your next round, go back to section 5 and set `NUM_QUERIES` much lower, then run sections 5 to 9 again.

Before you do, think about what you are trying to get from each query.